# 3. Cell annotation and Gaussian robustness

Run from the GitHub repository root with the **MetacellFormer-env** kernel after `02_embeddings.ipynb`. This notebook uses a fixed, class-balanced 64-cell subset of the released DC roster for cell annotation, then demonstrates one Gaussian condition (ratio 0.1, draw seed 42) and the clean-fitted robustness probe. It calls the public benchmark commands.

The archived DC input is described as TPM by its source.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

repo = Path.cwd().resolve()
assert (repo / "benchmarks/tasks/cell_annotation.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
work = repo / "example_outputs/DC"
roster_file = work / "example_roster.json"
source = data_root / "inputs/cell_annotation/DC.h5ad"
config = work / "runtime.json"
assert all(p.is_file() for p in (roster_file, source, config, work / "train.npz", work / "test.npz")), "Run notebook 02 first"
device = "cpu"  # Match notebook 02, or change to cuda:0.

In [ ]:
subprocess.run([sys.executable, "-m", "benchmarks.tasks.cell_annotation",
                "--train-embedding", str(work / "train.npz"),
                "--test-embedding", str(work / "test.npz"),
                "--roster", str(roster_file), "--model", "MetacellFormer",
                "--dataset", "DC", "--output-dir", str(work / "cell_annotation")],
               cwd=repo, check=True)
print(pd.read_csv(work / "cell_annotation/per_dataset_scores.tsv", sep="\t"))

In [ ]:
noise = work / "noise"
subprocess.run([sys.executable, "-m", "benchmarks.tasks.robustness", "prepare",
                "--input", str(source), "--roster", str(roster_file),
                "--dataset", "DC", "--ratios", "0.1", "--seeds", "42",
                "--output-dir", str(noise)], cwd=repo, check=True)
robust_embeddings = work / "robustness_embeddings"
robust_embeddings.mkdir(parents=True, exist_ok=True)
for name in ("train_clean", "clean", "gaussian_0.1_s42"):
    subprocess.run([sys.executable, "-m", "benchmarks.embeddings.extract_embeddings",
                    "--input", str(noise / f"{name}.h5ad"),
                    "--output", str(robust_embeddings / f"{name}.npz"),
                    "--config", str(config), "--model", "MetacellFormer",
                    "--task", "robustness", "--device", device, "--batch-size", "4"],
                   cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "benchmarks.tasks.robustness", "score",
                "--roster", str(roster_file), "--embedding-dir", str(robust_embeddings),
                "--model", "MetacellFormer", "--dataset", "DC",
                "--ratios", "0.1", "--seeds", "42",
                "--output-dir", str(work / "robustness_scores")], cwd=repo, check=True)
print(pd.read_csv(work / "robustness_scores/all_scores.tsv", sep="\t")[["probe_seed", "macro_f1", "clean_macro_f1", "retention"]])